In [1]:
from pyspark.sql.functions import col, avg, count, sum, round, when

# 1. Load Silver Tables
silver_assessments = spark.read.table("silver_assessments")
silver_courses = spark.read.table("silver_courses")
silver_students = spark.read.table("silver_students")

# --- a) Gold: Course Performance ---
# Aggregates metrics per course
gold_course_performance = silver_assessments.join(silver_courses, "course_id") \
    .groupBy("course_id", "course_name", "category") \
    .agg(
        count("student_id").alias("total_enrollments"),
        round((sum(when(col("completion_status") == "Completed", 1).otherwise(0)) / count("student_id")) * 100, 2).alias("completion_rate"),
        round(avg("score"), 2).alias("avg_score"),
        round(avg("time_spent_hours"), 2).alias("avg_time_spent_hours"),
        round(avg("feedback_rating"), 2).alias("avg_feedback_rating"),
        sum(when(col("certificate_issued") == True, 1).otherwise(0)).alias("certificate_issued_count"),
        round((sum(when(col("completion_status") == "Dropped", 1).otherwise(0)) / count("student_id")) * 100, 2).alias("drop_rate")
    )

# --- b) Gold: Student Engagement ---
# Aggregates metrics per student
gold_student_engagement = silver_assessments.join(silver_students, "student_id") \
    .groupBy("student_id", "subscription_type", "country") \
    .agg(
        count("course_id").alias("courses_enrolled"),
        sum(when(col("completion_status") == "Completed", 1).otherwise(0)).alias("courses_completed"),
        round(avg("score"), 2).alias("avg_score"),
        round(sum("time_spent_hours"), 2).alias("total_time_spent_hours"),
        sum(when(col("certificate_issued") == True, 1).otherwise(0)).alias("certificates_earned")
    )

# 2. Save as Managed Tables (for SQL Querying)
gold_course_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_course_performance")
gold_student_engagement.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_student_engagement")

# 3. Save as Physical Delta Files (in Files/gold folder)
gold_path = "Files/gold"
gold_course_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/course_performance_delta")
gold_student_engagement.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/student_engagement_delta")

# --- Output Verification ---
print("-" * 40)
print(f"gold_course_performance created: {gold_course_performance.count()} rows")
print(f"gold_student_engagement created: {gold_student_engagement.count()} rows")
print("-" * 40)
print("Gold Layer processing complete. Files saved to Files/gold.")

display(gold_course_performance.limit(5))

StatementMeta(, db418b81-11e2-4326-bb4f-2c48fec8ff85, 3, Finished, Available, Finished, False)

----------------------------------------
gold_course_performance created: 100 rows
gold_student_engagement created: 500 rows
----------------------------------------
Gold Layer processing complete. Files saved to Files/gold.


SynapseWidget(Synapse.DataFrame, 7100f391-ac91-4e62-9f3c-c58a50a19702)

In [2]:
# Save tables as Parquet for Snowflake ingestion
spark.table("silver_students").write.mode("overwrite").parquet("Files/export/students")
spark.table("silver_courses").write.mode("overwrite").parquet("Files/export/courses")
spark.table("silver_assessments").write.mode("overwrite").parquet("Files/export/assessments")
spark.table("gold_course_performance").write.mode("overwrite").parquet("Files/export/course_performance")
spark.table("gold_student_engagement").write.mode("overwrite").parquet("Files/export/student_engagement")

StatementMeta(, db418b81-11e2-4326-bb4f-2c48fec8ff85, 4, Finished, Available, Finished, False)

In [1]:
# Save tables as CSV for Snowflake ingestion
spark.table("silver_students").write.mode("overwrite").format("csv").option("header", "true").save("Files/export/students")
spark.table("silver_courses").write.mode("overwrite").format("csv").option("header", "true").save("Files/export/courses")
spark.table("silver_assessments").write.mode("overwrite").format("csv").option("header", "true").save("Files/export/assessments")
spark.table("gold_course_performance").write.mode("overwrite").format("csv").option("header", "true").save("Files/export/course_performance")
spark.table("gold_student_engagement").write.mode("overwrite").format("csv").option("header", "true").save("Files/export/student_engagement")

StatementMeta(, e65fe948-34bd-4eac-97cd-783ee19eb5a9, 3, Finished, Available, Finished, False)